### qwen-model

In [ ]:
## write the full model architecture here

In [86]:
## rope
import torch

def compute_rope_angles(head_dim, theta_base=10000, context_length=2048, dtype=torch.float32):
    
    assert head_dim // 2 == 0 ## head_dim should be  even

    index = torch.arange(0, head_dim, 2, dtype=dtype)
    inv_freq = 1.0 / (theta_base ** (2 * index / head_dim))

    ## compute positions
    positions = torch.arange(context_length, dtype=dtype) ## we are calculating the positions here  

    ## computing the angle 
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0) ### positions = 2048, 1  ** 1 ---- inv_freq = head_dim // 2 -----> angles = 2048 * 64

   
    cos = torch.cos(angles) ## 2048, 64
    sin = torch.sin(angles)  ## 2048, 64




In [ ]:
def apply_rope(x, cos, sin):

    B, T, H, D = x.shape

    x1 = x[..., ::2]   # [B, T, H, D/2]
    x2 = x[..., 1::2]  # [B, T, H, D/2]

    # Handle both training and inference
    if cos.dim() == 2:  # training
        cos = cos.unsqueeze(0).unsqueeze(2)  # [1, T, 1, D/2]
        sin = sin.unsqueeze(0).unsqueeze(2)
    else:               # inference (single position)
        cos = cos[None, None, None, :]       # [1, 1, 1, D/2]
        sin = sin[None, None, None, :]

    x_even = x1 * cos - x2 * sin
    x_odd  = x1 * sin + x2 * cos

    x_out = torch.stack([x_even, x_odd], dim=-1).flatten(-2)

    return x_out

In [ ]:
## RMSNorm
import torch
import torch.nn as nn

class RMSNorm(nn.module):
    def __init__(self, emb, eps=1e-6):
        super().__init__()

        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb))

    def forward(self, x):
        square = torch.square(x)
        sq_mean = square.mean(dim=-1, keepdim=True)  #b,t,1
        value = sq_mean +  self.eps
        rms_value = torch.sqrt(value)
        normalized_value = x / rms_value
        value = normalized_value * self.weight
        return value 


    

In [ ]:
import torch
import torch.nn as nn

class GroupQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, kv_heads, head_dim, dtype=torch.float32):
        super().__init__()

        self.d_in = d_in
        self.num_heads = num_heads
        self.kv_heads = kv_heads
        self.group_size = num_heads / kv_heads
        self.head_dim = head_dim

        self.d_out = num_heads * head_dim


        ## projections 

        self.w_query = nn.Linear(self.d_in, self.d_out, dtype=dtype)
        self.w_keys = nn.Linear(self.d_in, self.kv_heads * self.head_dim, dtype=dtype)
        self.w_values = nn.Linear(self.d_in, self.kv_heads * self.head_dim, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, self.d_in, dtype=dtype)


        ## norms
        self.q_norm = RMSNorm(self.d_out)
        self.k_norm = RMSNorm(kv_heads * head_dim)



    def forward(self, x, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        ## apply projections 
        queries = self.w_query(x) # (b, t, num_heads * head_dim)
        keys = self.w_keys(x)   # (b, t, kv_heads * head_dim)
        values = self.w_values(x) # (b, t, kv_heads * head_dim)


        ## reshape to heads/kv groups
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1,2)  # (b, num_heads, num_tokens, head_dim)
        keys_new = keys.view(b, num_tokens, self.kv_heads, self.head_dim).transpose(1,2)  # (b, kv_heads, num_tokens, head_dim)
        values_new = values.view(b, num_tokens, self.kv_heads, self.head_dim).transpose(1,2)



        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys_new = apply_rope(keys, cos, sin, offset = start_pos)

        ## cache
        if cache is not None: 
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
        else: 
            start_pos = 0
            keys, values = keys_new, values_new
        next_cache = (keys, values)


        ## expand k and v to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)


        ## attention....

